# NLP Mastery Journey — Module 8: Fine-Tuning Pretrained Transformers

Module 7 ended on the real industry workflow: start from a model someone else already spent millions of dollars pretraining, and adapt it to your task instead of training from zero. This module is that adaptation step, end to end — using Hugging Face's `transformers` library, the same tooling used inside research teams at Google, Meta, and countless startups.

### What this notebook teaches
| # | Topic | Why it matters |
|---|-------|------------------|
| 1 | Prompting vs. fine-tuning vs. RAG | The decision every real LLM project starts with |
| 2 | Loading & using a pretrained model | `AutoTokenizer` / `AutoModel*` — the standard entry point |
| 3 | Fine-tuning with the `Trainer` API | The high-level, production-standard training workflow |
| 4 | A manual PyTorch fine-tuning loop | What `Trainer` is actually doing underneath |
| 5 | Freezing layers / partial fine-tuning | Full fine-tune vs. feature-extraction vs. partial unfreeze |
| 6 | Learning rate warmup & scheduling | Why fine-tuning needs a different LR strategy than training from scratch |
| 7 | LoRA (Low-Rank Adaptation) | The modern, dominant technique for cheap, scalable fine-tuning |
| 8 | Other PEFT methods | Adapters, prefix tuning — the landscape beyond LoRA |
| 9 | Token classification (NER) fine-tuning | The other major fine-tuning shape, beyond sentence classification |
| 10 | Production: evaluation, forgetting, serving | Shipping a fine-tuned model safely |

### How to use this notebook
- Parts 6 and 7 (LR scheduling, LoRA) are demonstrated in **plain NumPy/PyTorch math** — genuinely runnable, no downloads — so you see the actual mechanics, not just a description.
- Everything touching a real pretrained model (`transformers`/`datasets` downloads) is written correctly but commented out, exactly like the network-dependent cells in Modules 1, 3, 4, and 7 — uncomment and run with internet access.
- **🔀 Alternatives** and **📋 Copy-paste template** callouts continue as before.


## 0. Setup

In [ ]:
# %pip install torch --index-url https://download.pytorch.org/whl/cpu
# %pip install transformers datasets accelerate evaluate peft scikit-learn matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

print("Setup note: uncomment the pip install lines above the first time you run this.")


## Part 1 — Prompting vs. Fine-Tuning vs. RAG

Before writing any training code, the actual first question on any real project: **do you even need to fine-tune?** This is one of the most consequential decisions in building an LLM-based product, and teams regularly over-engineer by reaching for fine-tuning too early.

| Approach | What it is | Best when... | Cost/complexity |
|----------|------------|-----------------|---------------------|
| **Zero/few-shot prompting** | Describe the task (+ a few examples) directly in the prompt to a general-purpose model | The task is well within the model's general capabilities; you need to move fast; data is scarce | Lowest — no training infrastructure at all |
| **RAG (Retrieval-Augmented Generation)** | Retrieve relevant documents at query time and include them in the prompt | The model needs access to information it wasn't trained on, or that changes frequently (your company's docs, today's data) | Low-medium — needs a retrieval/embedding pipeline (Module 4), no model training |
| **Fine-tuning** | Update the model's own weights on labeled examples of your specific task | You need a specific output FORMAT reliably, a narrow/specialized domain vocabulary, or lower per-request latency/cost than a large general model at scale | Highest — needs labeled data, training infrastructure, evaluation, versioning |

### A concrete rule of thumb many production teams actually use
1. **Start with prompting** a strong general model. It's the fastest to test, and often just works.
2. **Add RAG** if the model's failures are about *missing information* (it doesn't know something), not about *how* it responds.
3. **Fine-tune** if the model's failures are about *behavior/format/style* that prompting can't reliably fix, or if you need a smaller, cheaper, faster model for a narrow task at high volume (this connects directly to Module 5's point: sometimes the answer is a small, cheap, specialized model, not a bigger general one).

This module covers option 3 — but knowing when NOT to reach for it is itself a genuinely valuable, production-relevant skill.


## Part 2 — Loading & Using a Pretrained Model

The Hugging Face "Auto classes" are the standard entry point: `AutoTokenizer` and `AutoModelFor*` automatically load the right tokenizer/architecture for whatever pretrained checkpoint name you give them — you rarely need to know the exact underlying class.


In [ ]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
#
# model_name = "distilbert-base-uncased"   # a smaller, faster BERT variant —
#                                           # a common, sensible default to start
#                                           # experimenting with before reaching
#                                           # for a larger model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(
#     model_name,
#     num_labels=2,   # binary classification — swap to however many classes YOUR task has
# )
#
# # Running inference right away, with NO fine-tuning at all, just to see the shapes:
# inputs = tokenizer("This movie was fantastic!", return_tensors="pt")
# outputs = model(**inputs)
# print(outputs.logits.shape)   # (1, num_labels) — but these predictions are
#                                # MEANINGLESS right now: the classification head
#                                # was just randomly initialized and has never
#                                # been trained on YOUR task yet — that's what
#                                # the rest of this module fixes

print("Model + tokenizer loading pattern shown above — needs internet the first run to "
      "download the pretrained checkpoint (~250MB for distilbert-base-uncased).")

# 🔀 Common starting checkpoints, by task
# | Checkpoint                        | Good default for...                          |
# |----------------------------------------|---------------------------------------------------|
# | distilbert-base-uncased                  | Fast, cheap classification baseline                  |
# | bert-base-uncased                          | Stronger classification, still widely supported        |
# | roberta-base                                 | Often a few points stronger than BERT on the same task   |
# | bert-base-multilingual-cased                   | Non-English or multilingual text                          |


## Part 3 — Fine-Tuning with the `Trainer` API

Hugging Face's `Trainer` is the high-level, production-standard way to fine-tune: it handles the training loop, evaluation, checkpointing, logging, and mixed-precision/multi-GPU details for you. This is genuinely what most industry fine-tuning code looks like day to day.


In [ ]:
# from datasets import Dataset
# from transformers import TrainingArguments, Trainer
# import numpy as np
#
# # ── Step 1: your labeled data, as a Hugging Face Dataset ────────────────────
# # (Reuse the sentiment reviews + labels from Module 5/6 here, or your own CSV
# # loaded via Module 1's pd.read_csv() and converted with Dataset.from_pandas())
# train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
# eval_dataset = Dataset.from_dict({"text": eval_texts, "label": eval_labels})
#
# # ── Step 2: tokenize everything up front ─────────────────────────────────────
# def tokenize_function(examples):
#     return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
#
# train_dataset = train_dataset.map(tokenize_function, batched=True)
# eval_dataset = eval_dataset.map(tokenize_function, batched=True)
#
# # ── Step 3: define how to compute metrics during evaluation ─────────────────
# import evaluate
# accuracy_metric = evaluate.load("accuracy")
# f1_metric = evaluate.load("f1")
#
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)
#     return {
#         "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
#         "f1": f1_metric.compute(predictions=predictions, references=labels)["f1"],
#     }
#
# # ── Step 4: TrainingArguments — the hyperparameters that matter most ────────
# training_args = TrainingArguments(
#     output_dir="./fine_tuned_model",
#     learning_rate=2e-5,           # small! see Part 6 for why fine-tuning LRs
#                                   # are 10-100x smaller than training-from-scratch LRs
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,            # fine-tuning usually needs FAR fewer epochs
#                                    # than training from scratch — 2-4 is typical
#     weight_decay=0.01,              # L2 regularization, same idea as Module 5's
#                                     # Logistic Regression C parameter, inverted
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,    # automatically keep the best checkpoint,
#                                     # not just whichever epoch finished last
#     metric_for_best_model="f1",
# )
#
# # ── Step 5: put it all together and train ────────────────────────────────────
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     compute_metrics=compute_metrics,
# )
# trainer.train()
#
# final_metrics = trainer.evaluate()
# print(final_metrics)

print("Full Trainer-based fine-tuning workflow shown above — the standard production pattern.")


## Part 4 — A Manual PyTorch Fine-Tuning Loop

Worth seeing once: `Trainer` is convenient, but it's not magic — this is (roughly) what it does underneath, and it's the exact same training loop shape from Module 6, applied to a pretrained model instead of one you built from scratch.


In [ ]:
# import torch
# from torch.utils.data import DataLoader
# from torch.optim import AdamW
# from transformers import get_linear_schedule_with_warmup
#
# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# optimizer = AdamW(model.parameters(), lr=2e-5)
#
# num_epochs = 3
# num_training_steps = num_epochs * len(train_loader)
# scheduler = get_linear_schedule_with_warmup(
#     optimizer, num_warmup_steps=int(0.1 * num_training_steps), num_training_steps=num_training_steps
# )   # see Part 6 for exactly what this schedule looks like and why it matters
#
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
#
# for epoch in range(num_epochs):
#     model.train()
#     for batch in train_loader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         optimizer.zero_grad()
#
#         outputs = model(**batch)          # Hugging Face models compute the loss
#         loss = outputs.loss                # INTERNALLY when you pass `labels` in the batch —
#                                            # no need to write your own loss function
#         loss.backward()
#
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # Module 6's
#                                                                            # gradient-clipping
#                                                                            # pattern, standard here too
#         optimizer.step()
#         scheduler.step()                   # advance the LR schedule EVERY step, not every epoch
#
#     print(f"Epoch {epoch+1} complete.")

print("Manual training-loop pattern shown above — notice it's the SAME shape as Module 6's "
      "train_model(), just with a pretrained model and a warmup scheduler added.")


## Part 5 — Freezing Layers: Full Fine-Tune vs. Feature Extraction vs. Partial

Three levels of how much of the pretrained model you actually update:

1. **Full fine-tuning**: every parameter is trainable. Best accuracy potential, most compute/memory, highest risk of *catastrophic forgetting* (the model overwriting useful general knowledge it learned during pretraining).
2. **Feature extraction (fully frozen)**: freeze the ENTIRE pretrained model, only train a new classifier head on top. Fastest, cheapest, safest against forgetting — but leaves the most performance on the table.
3. **Partial fine-tuning**: freeze the lower layers (which tend to capture general, reusable language patterns) and only unfreeze the upper layers + head (which tend to capture more task-specific patterns). A common middle ground.


In [ ]:
# ── Freezing parameters in PyTorch ────────────────────────────────────────────

# # Option 1: freeze EVERYTHING except the classification head
# for param in model.base_model.parameters():   # "base_model" is the pretrained
#     param.requires_grad = False                # backbone, excluding the task-specific head
# # only the (randomly-initialized, newly-added) classifier head remains trainable
#
# # Option 2: freeze only the FIRST N layers, leave the rest trainable
# num_layers_to_freeze = 6
# for layer in model.base_model.encoder.layer[:num_layers_to_freeze]:
#     for param in layer.parameters():
#         param.requires_grad = False
#
# # Always sanity-check how many parameters are ACTUALLY trainable:
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total_params = sum(p.numel() for p in model.parameters())
# print(f"Trainable: {trainable_params:,} / {total_params:,} "
#       f"({100 * trainable_params / total_params:.1f}%)")

print("Layer-freezing patterns shown above — always print the trainable-parameter "
      "count afterward to confirm you froze what you intended to.")

# 🔀 Which to pick
# | Situation                                       | Choose                |
# |------------------------------------------------------|--------------------------|
# | Plenty of labeled data, task is quite different from pretraining| Full fine-tuning|
# | Very little labeled data                                | Feature extraction (safest against overfitting)|
# | Medium data, want a balance of speed and accuracy          | Partial fine-tuning |
# | Need to fine-tune a LARGE model cheaply                      | LoRA (Part 7) — usually beats naive freezing |


## Part 6 — Learning Rate Warmup & Scheduling

Fine-tuning uses learning rates roughly **10-100x smaller** than training from scratch (typically 1e-5 to 5e-5, vs. 1e-3 to 1e-2) — the pretrained weights already encode a huge amount of useful structure, and large updates risk destroying it (more catastrophic forgetting). On top of a small learning rate, a **warmup** period (start near zero, ramp up, then decay) further protects against large, destabilizing updates right at the start of training, when the newly-added classifier head's random weights produce the largest, noisiest gradients.


In [ ]:
# ── Visualizing a linear-warmup-then-decay schedule (the standard for fine-tuning) ─

def linear_warmup_schedule(num_training_steps, num_warmup_steps, base_lr):
    steps = np.arange(num_training_steps)
    lrs = np.zeros(num_training_steps)

    # Phase 1: linear WARMUP from 0 up to base_lr
    warmup_mask = steps < num_warmup_steps
    lrs[warmup_mask] = base_lr * (steps[warmup_mask] / max(1, num_warmup_steps))

    # Phase 2: linear DECAY from base_lr back down to 0
    decay_mask = ~warmup_mask
    remaining_steps = num_training_steps - num_warmup_steps
    lrs[decay_mask] = base_lr * (1 - (steps[decay_mask] - num_warmup_steps) / max(1, remaining_steps))

    return lrs

num_training_steps = 1000
num_warmup_steps = 100   # a common convention: ~10% of total steps
base_lr = 2e-5

lrs = linear_warmup_schedule(num_training_steps, num_warmup_steps, base_lr)

plt.figure(figsize=(8, 4))
plt.plot(lrs)
plt.axvline(num_warmup_steps, color="gray", linestyle="--", label="end of warmup")
plt.xlabel("Training step")
plt.ylabel("Learning rate")
plt.title("Linear Warmup + Linear Decay Schedule")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Peak LR: {lrs.max():.2e} at step {lrs.argmax()}")


In [ ]:
# 📋 COPY-PASTE TEMPLATE — the real thing, via Hugging Face's transformers:
#
# from transformers import get_linear_schedule_with_warmup
# scheduler = get_linear_schedule_with_warmup(
#     optimizer,
#     num_warmup_steps=int(0.1 * total_steps),   # ~10% of total steps is a common default
#     num_training_steps=total_steps,
# )
#
# 🔀 Other common schedules
# | Schedule                | Shape                                                    |
# |----------------------------|---------------------------------------------------------|
# | Linear warmup + linear decay | What we plotted above — a strong, simple default        |
# | Cosine schedule                | Smoother decay curve (follows a cosine curve down to 0)   |
# | Constant with warmup              | Warms up, then stays FLAT — simpler, sometimes less optimal|

print("Real HF scheduler usage and alternatives shown above.")


## Part 7 — LoRA (Low-Rank Adaptation): the Modern Default

Full fine-tuning of a large model means storing gradients and optimizer state for **every single parameter** — for a multi-billion-parameter model, that's an enormous memory/compute cost, and you end up with an entirely new full-size copy of the model per task. **LoRA** is the technique that made fine-tuning large models cheap and practical, and it's genuinely the industry-standard approach at scale today (used constantly across big tech and open-source fine-tuning alike).

### The core idea
Instead of updating a weight matrix $W$ (shape $d \times d$) directly, LoRA **freezes $W$ entirely** and learns a small *update* to it, expressed as the product of two much smaller matrices:

$$W' = W + BA \qquad \text{where } B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times d},\ r \ll d$$

$r$ (the "rank") is typically 4-64 — tiny compared to $d$, which is often 768-12288+ in real models. This works because the *useful update* needed to adapt a pretrained model to a new task tends to have much lower "intrinsic rank" than the full weight matrix — empirically, you don't need to move $W$ in every possible direction, just a small, structured subspace of directions.


In [ ]:
# ── Demonstrating the actual parameter savings, with real numbers ────────────

d = 768        # a typical BERT-base hidden dimension
r = 8          # a common LoRA rank

full_finetune_params = d * d                    # updating the WHOLE weight matrix
lora_params = d * r + r * d                       # updating ONLY the two small matrices A and B

print(f"Full fine-tuning this ONE weight matrix: {full_finetune_params:,} parameters")
print(f"LoRA (rank={r}):                          {lora_params:,} parameters")
print(f"Reduction: {full_finetune_params / lora_params:.1f}x fewer trainable parameters")

# And this compounds: a real model has DOZENS of such matrices (every attention
# and feedforward weight matrix, across every layer) — LoRA applies this same
# trick to each one, typically cutting TOTAL trainable parameters by 100-1000x
# compared to full fine-tuning, while recovering most of full fine-tuning's
# accuracy on many tasks.


In [ ]:
# ── A tiny, from-scratch LoRA layer in NumPy — the actual forward-pass math ──

class LoRALinear:
    """
    A minimal, from-scratch illustration of how a LoRA-adapted linear layer
    computes its output. Real implementations (see the peft library template
    below) integrate this directly into PyTorch's autograd; this version
    exists purely so you can see the math with nothing hidden.
    """
    def __init__(self, in_dim, out_dim, rank, alpha=16):
        # The ORIGINAL pretrained weight — FROZEN, never updated during LoRA training
        self.W = np.random.randn(out_dim, in_dim) * 0.01

        # The LoRA update matrices — these ARE trained; B starts at all zeros
        # so that at the very start of training, W' == W exactly (no
        # disruption to the pretrained model before any learning has happened)
        self.A = np.random.randn(rank, in_dim) * 0.01
        self.B = np.zeros((out_dim, rank))

        self.scaling = alpha / rank   # a scaling factor that keeps the update's
                                      # magnitude comparable across different rank choices

    def forward(self, x):
        original_output = x @ self.W.T                       # the frozen pretrained computation
        lora_update = x @ self.A.T @ self.B.T * self.scaling   # the small, LEARNED adjustment
        return original_output + lora_update                   # W' @ x, computed WITHOUT ever
                                                                # materializing W' explicitly

lora_layer = LoRALinear(in_dim=768, out_dim=768, rank=8)
sample_input = np.random.randn(4, 768)   # (batch=4, in_dim=768)
output = lora_layer.forward(sample_input)
print("Output shape:", output.shape)   # (4, 768) — same as a normal Linear layer would produce

# Confirming B starting at zero means B initially contributes nothing:
print("Initial LoRA update magnitude (should be exactly 0):",
      np.abs(sample_input @ lora_layer.A.T @ lora_layer.B.T).sum())


In [ ]:
# 📋 COPY-PASTE TEMPLATE — real LoRA fine-tuning via Hugging Face's `peft` library
#
# from peft import LoraConfig, get_peft_model, TaskType
#
# lora_config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,       # matches your model type (SEQ_CLS, TOKEN_CLS, CAUSAL_LM, etc.)
#     r=8,                                # the rank from the math above
#     lora_alpha=16,                       # the scaling factor from the math above
#     lora_dropout=0.1,
#     target_modules=["query", "value"],    # WHICH weight matrices get a LoRA adapter —
#                                           # attention's query/value projections are a
#                                           # very common, effective default choice
# )
#
# peft_model = get_peft_model(model, lora_config)
# peft_model.print_trainable_parameters()
# # e.g. "trainable params: 294,912 || all params: 66,955,010 || trainable%: 0.44"
#
# # From here, train EXACTLY as in Part 3/4 — peft_model is a drop-in
# # replacement, Trainer/your manual loop don't need to know anything changed.
# trainer = Trainer(model=peft_model, args=training_args, train_dataset=train_dataset, ...)
# trainer.train()
#
# # Saving is tiny too — only the small adapter weights get saved, not a
# # full copy of the base model:
# peft_model.save_pretrained("./lora_adapter")   # often just a few MB, vs. hundreds of MB-GB

print("Real peft/LoRA workflow shown above — notice trainable% is typically under 1%.")


## Part 8 — Other Parameter-Efficient Fine-Tuning (PEFT) Methods

LoRA is the dominant choice today, but it's part of a broader family collectively called **PEFT** — all sharing the same goal: adapt a large frozen model while training only a small number of new parameters.

| Method | Idea | Trade-off |
|--------|------|-------------|
| **LoRA** | Low-rank update matrices added to existing weights | Best balance of simplicity, performance, and adoption — the default starting point |
| **QLoRA** | LoRA on top of a **quantized** (4-bit) frozen base model | Even lower memory — lets you fine-tune genuinely huge models on a single consumer GPU |
| **Adapters** | Small new feedforward layers INSERTED between existing layers | Conceptually simple, slightly adds inference latency (extra layers to run through) |
| **Prefix/Prompt tuning** | Learn a small set of "virtual tokens" prepended to the input, rest of the model fully frozen | Extremely parameter-efficient, but generally weaker than LoRA on harder tasks |

🔀 **Practical default**: start with LoRA (or QLoRA if memory-constrained). It's the best-supported, most battle-tested option, and rarely the wrong first choice.


## Part 9 — Fine-Tuning for Token Classification (NER)

Module 6, Part 7 introduced sequence labeling conceptually; here's the real fine-tuning shape for it. The key extra wrinkle: subword tokenization (Module 7, Part 7) can split ONE word into MULTIPLE tokens, so labels need to be realigned to match the tokenizer's output.


In [ ]:
# from transformers import AutoTokenizer, AutoModelForTokenClassification
#
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# model = AutoModelForTokenClassification.from_pretrained("bert-base-uncased", num_labels=9)
# # 9 labels here is a common NER tag-set size, e.g. the CoNLL-2003 scheme:
# # O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, B-MISC, I-MISC
#
# def tokenize_and_align_labels(examples):
#     """
#     📋 COPY-PASTE TEMPLATE — the standard label-realignment function for
#     any token-classification fine-tuning project. "New York" might become
#     two subword tokens; this ensures the LOCATION label lines up correctly
#     with EACH of them.
#     """
#     tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
#     all_labels = []
#     for i, labels in enumerate(examples["ner_tags"]):
#         word_ids = tokenized.word_ids(batch_index=i)   # maps each SUBWORD token back
#                                                         # to its ORIGINAL word index
#         previous_word_idx = None
#         label_ids = []
#         for word_idx in word_ids:
#             if word_idx is None:
#                 label_ids.append(-100)             # -100 = "ignore this position in the loss" —
#                                                     # used for special tokens like [CLS]/[SEP]
#             elif word_idx != previous_word_idx:
#                 label_ids.append(labels[word_idx])   # first subword of a word gets the REAL label
#             else:
#                 label_ids.append(-100)                # subsequent subwords of the SAME word are
#                                                        # ignored in the loss (a common convention;
#                                                        # some projects instead repeat the label)
#             previous_word_idx = word_idx
#         all_labels.append(label_ids)
#     tokenized["labels"] = all_labels
#     return tokenized

print("Token-classification fine-tuning + label-alignment pattern shown above — "
      "the -100 ignore-index convention is standard across virtually all HF NER code.")


## Part 10 — Production: Evaluation, Catastrophic Forgetting, and Serving

### Catastrophic forgetting
Fine-tuning too aggressively (too high an LR, too many epochs, full fine-tuning on too little data) can make the model forget general capabilities it had before fine-tuning, even ones unrelated to your task. Symptoms: the model gets great at your narrow task but noticeably worse at everything else. Mitigations, in order of how often they're used: lower the learning rate, fewer epochs, LoRA/partial freezing (fewer parameters change = less to forget), and evaluating on a broader benchmark, not just your task's own test set, before shipping.

### Evaluating a fine-tuned model properly
Beyond your task metric (accuracy/F1/etc.), production teams typically also check: performance on a **held-out set from a DIFFERENT time period** than training data (catches distribution shift early), a handful of adversarial/edge-case examples written by hand, and — increasingly — an LLM-as-judge evaluation for open-ended generation tasks where a single accuracy number doesn't capture quality well.

### Serving a fine-tuned model
Training is a small fraction of a model's total lifecycle cost — most of the cost is **inference**, running at scale, forever. Standard production serving tools: **vLLM** and **Text Generation Inference (TGI)** for efficient generation serving (both implement KV-caching and batching optimizations from Module 7 automatically), or a managed endpoint if you don't want to operate serving infrastructure yourself.


In [ ]:
# ── 📋 COPY-PASTE TEMPLATE: saving and reloading a fine-tuned model for production ─

# model.save_pretrained("./my_fine_tuned_model")
# tokenizer.save_pretrained("./my_fine_tuned_model")   # ALWAYS save the tokenizer
#                                                       # alongside the model — a
#                                                       # mismatched tokenizer silently
#                                                       # produces garbage predictions
#
# # Later, in a completely separate process:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# loaded_tokenizer = AutoTokenizer.from_pretrained("./my_fine_tuned_model")
# loaded_model = AutoModelForSequenceClassification.from_pretrained("./my_fine_tuned_model")
# loaded_model.eval()
#
# inputs = loaded_tokenizer("a brand new review to classify", return_tensors="pt")
# with torch.no_grad():
#     logits = loaded_model(**inputs).logits
# prediction = logits.argmax(dim=-1).item()
#
# # Optionally: push to the Hugging Face Hub for easy sharing/versioning across a team
# # model.push_to_hub("your-username/my-fine-tuned-model")
# # tokenizer.push_to_hub("your-username/my-fine-tuned-model")

print("Save/load/share patterns shown above — always ship the tokenizer WITH the model.")


## Recap & What's Next

You now have the full decision framework (prompting vs. RAG vs. fine-tuning) plus the complete fine-tuning toolkit: the `Trainer` API, a manual training loop for understanding what it does underneath, layer-freezing strategies, learning-rate warmup scheduling, and — the modern industry default — LoRA, implemented from scratch so you've seen the actual low-rank math, not just a library call. You also covered the production side that a lot of tutorials skip: catastrophic forgetting, proper evaluation, and serving.

### Try this before the next lesson
1. Take your Module 5/6 sentiment dataset and write out the full `Trainer`-based fine-tuning code for `distilbert-base-uncased`, even if you can't run it right now — the exercise of filling in every argument correctly is where this sticks.
2. Compute the LoRA parameter-reduction ratio (Part 7) for a rank of 4 and a rank of 64 on a `d=1024` weight matrix, and think about what that trade-off means for accuracy vs. efficiency.
3. Write out, in your own words, a real scenario from something you've used where RAG would be the right call instead of fine-tuning — and one where fine-tuning would be right instead of RAG.

### Next lesson in your NLP mastery path
**Module 9: Retrieval-Augmented Generation (RAG) and LLM Application Patterns** — building a real retrieval pipeline end to end (the embeddings from Module 4, a vector index from Module 4's semantic search, and a generation step on top), chunking strategies, evaluating RAG quality, and the broader patterns (agents, tool use, structured output) that real LLM products are built from today.
